# System D candidates for the rating sheets (Phase 5, weight tuning)

Writes the 400 System D poems the reranker ratings need: 10 for each of the top 2 rhyme plans of the 20 System D
prompts in `data/rerank_ratings/prompts.jsonl`. The prompts were made locally, because the rhyme planner needs the
GloVe files, which are not in the repo.

**On Colab:** Runtime → Change runtime type → **T4 GPU**, then Run all. It needs the trained adapter in
`MyDrive/CS425/checkpoints/gen_D_1.5b_best/` (put there by `generation/train_qwen.ipynb`). About 10 minutes on a T4.

**Output:** `candidates_D.jsonl`, saved to `MyDrive/CS425/rerank_ratings/` and downloaded by the last cell. Put it in
`data/rerank_ratings/` in the project, then run `python reranker/rating_pool.py sheets` locally.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/SchoolWork-awork2103/CS425-Project.git"
BRANCH = "feat/phase0"
DRIVE_CKPT_DIR = "/content/drive/MyDrive/CS425/checkpoints"
DRIVE_OUT_DIR = "/content/drive/MyDrive/CS425/rerank_ratings"

from google.colab import drive
drive.mount("/content/drive")
root = "/content/CS425-Project"
if os.path.exists(root):
    subprocess.run(["git", "-C", root, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, root], check=True)
# Colab ships an old torchao that recent peft refuses to load alongside; nothing here uses it
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", "torchao"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "transformers>=4.56", "sentencepiece",
                "pyyaml", "pronouncing", "wordfreq"], check=True)
os.chdir(root)

In [ ]:
# everything the run needs, checked before the model loads
import torch
problems = []
if not torch.cuda.is_available():
    problems.append("no GPU: Runtime -> Change runtime type -> T4 GPU, then Run all again")
if not os.path.exists(os.path.join(DRIVE_CKPT_DIR, "gen_D_1.5b_best", "adapter_config.json")):
    problems.append(f"no adapter in {DRIVE_CKPT_DIR}/gen_D_1.5b_best/ (train it with generation/train_qwen.ipynb)")
if not os.path.exists("data/rerank_ratings/prompts.jsonl") or not os.path.exists("reranker/rating_pool.py"):
    problems.append(f"the {BRANCH} branch has no rating prompts yet: commit and push reranker/rating_pool.py, "
                    "config.yaml and data/rerank_ratings/prompts.jsonl first")
if problems:
    raise SystemExit("\n".join(problems))
print("ready:", torch.cuda.get_device_name())

In [ ]:
!python reranker/rating_pool.py generate --system D --ckpt-dir {DRIVE_CKPT_DIR} --out-dir {DRIVE_OUT_DIR}

In [ ]:
from google.colab import files
files.download(os.path.join(DRIVE_OUT_DIR, "candidates_D.jsonl"))